# V1 — 04: Evaluasi Final + Demo


## Bootstrap
Sel env+config+helpers disalin dari `01_setup` agar notebook ini mandiri pasca-restart kernel. Jalankan semua sel Bootstrap sebelum sel tahap.

In [ ]:
# === SECTION:setup ===
# ============================================================
# 1. Lingkungan: Colab / lokal / JupyterLab + dua root
# ============================================================
import os, sys, time, json, re, shutil, glob, math, random, logging, codecs as cs
from datetime import datetime
from types import SimpleNamespace

def _detect_env():
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        pass
    if os.path.exists("/content"):
        return "colab"
    if os.environ.get("JUPYTERHUB_USER") or os.environ.get("JPY_PARENT_PID"):
        return "jupyterlab"
    try:
        import jupyter_core  # noqa
        return "jupyterlab"
    except ImportError:
        return "local"

ENV = _detect_env()

def _find_repo_root(start):
    cur = os.path.abspath(start)
    for _ in range(6):
        if (os.path.isdir(os.path.join(cur, "Extract-Classify-ACOS"))
                and os.path.isdir(os.path.join(cur, "ACOS-BERT"))):
            return cur
        cur = os.path.dirname(cur)
    return None

USE_DRIVE = False  # set True di Colab bila sesi/checkpoint ingin di Drive
if ENV == "colab" and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    _drive_acos = "/content/drive/MyDrive/ACOS-ASLI"
    REPO_ROOT = _drive_acos if os.path.isdir(_drive_acos) else _find_repo_root(".")
else:
    REPO_ROOT = _find_repo_root(".") or os.path.abspath(".")
if REPO_ROOT is None:
    raise RuntimeError("Repo root (berisi Extract-Classify-ACOS/ + ACOS-BERT/) tidak ditemukan")

bert_root = os.path.join(REPO_ROOT, "ACOS-BERT")
acos_root = REPO_ROOT
extract_dir = os.path.join(REPO_ROOT, "Extract-Classify-ACOS")
for _p in (extract_dir, bert_root):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import acos_en, acos_en.upstream, acos_en.taxonomy, acos_en.datafiles, acos_en.selftest
acos_en.upstream.ensure_path(acos_root=acos_root)
import colab_utils  # helper sesi/plot/metrik upstream (read-only)

print(f"ENV={ENV} | REPO_ROOT={REPO_ROOT}")
print(f"bert_root={bert_root} (ditulis) | extract_dir={extract_dir} (dibaca saja)")


In [ ]:
# === SECTION:setup ===
# ============================================================
# 2. Konfigurasi run (ulang setiap restart kernel)
# ============================================================
DOMAIN = "rest16"        # "rest16" (13 kat -> 39 label) atau "laptop" (121 kat -> 363 label)
BACKBONE = "bert-en"     # tetap; V1 hanya mendukung bert-base-uncased
MAX_SEQ_LENGTH = 128
STEP1_BATCH_SIZE = 32
STEP2_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
STEP1_LR = 2e-5
STEP2_LR = 5e-5
NUM_EPOCHS = 15          # paper: 30; 15 optimal untuk sesi GPU Colab/JupyterLab
SEED = 42
PATIENCE = 5
MIN_EPOCHS_BEFORE_STOP = 5
GRAD_ACC = 1
WARMUP = 0.1
FORCE_RETRAIN_STEP1 = False
FORCE_RETRAIN_STEP2 = False
USE_AMP = False          # True bila torch>=1.6 + GPU (opsional, hemat VRAM)

N_CATSENTI = {"rest16": 39, "laptop": 363}
assert DOMAIN in N_CATSENTI, DOMAIN
print(f"DOMAIN={DOMAIN} | num_labels_step2={N_CATSENTI[DOMAIN]} | SEED={SEED}")


In [ ]:
# === SECTION:setup ===
# ============================================================
# 3. Helper runtime (progres, _prf, tulis CSV/MD)
# ============================================================
import pandas as pd

class step_stage:
    def __init__(self, title, total_steps=None):
        self.title, self.total, self.n, self.t0 = title, total_steps, 0, None
    def __enter__(self):
        self.t0 = time.time()
        print("=" * 78); print(f">> {self.title}"); print("=" * 78, flush=True)
        return self
    def step(self, msg):
        self.n += 1
        tag = f"{self.n}/{self.total}" if self.total else str(self.n)
        print(f"   [{tag}] {time.time()-self.t0:7.1f}s  {msg}", flush=True)
    def note(self, msg):
        print(f"        {msg}", flush=True)
    def __exit__(self, t, e, tb):
        print(("OK " if t is None else f"GAGAL: {e} ") + f"{self.title} ({time.time()-self.t0:.1f}s)", flush=True)
        return False

def require_vars(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(f"Variabel {missing} belum ada — jalankan sel 1-2 lebih dulu.")

def _prf(tp, fp, fn):
    p = tp/(tp+fp) if tp+fp else 0.0
    r = tp/(tp+fn) if tp+fn else 0.0
    return p, r, (2*p*r/(p+r) if p+r else 0.0)

def write_df_csv_md(df, csv_path, md_path=None, title=""):
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df.to_csv(csv_path, index=False)
    if md_path:
        os.makedirs(os.path.dirname(md_path), exist_ok=True)
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(f"# {title}\n\n" + df.to_markdown(index=False) + "\n")
    return csv_path

def history_display_frame(history):
    return pd.DataFrame(history) if history else pd.DataFrame()

print("helper siap: step_stage, require_vars, write_df_csv_md")


In [ ]:
# === SECTION:setup ===
# ============================================================
# 4. Gerbang verifikasi torch-free (merah = berhenti)
# ============================================================
require_vars("step_stage", "bert_root", "acos_root", "extract_dir")
from acos_en import default_paths
paths = default_paths(bert_root, acos_root)
gates = acos_en.selftest.run_gates(DOMAIN, paths, raise_on_fail=True)
for g, r in gates.items():
    print(f"[LULUS] {g}: {r['detail']}")
print(f"{len(gates)}/{len(gates)} gate torch-free hijau")


In [ ]:
# === SECTION:setup ===
# ============================================================
# 5. Backbone bert-en + Gate-1 (bobot benar-benar termuat)
# ============================================================
require_vars("step_stage", "bert_root", "BACKBONE", "paths")
import acos_en.checkpoint as acos_ckpt
voc = acos_ckpt.ensure_vocab(paths["backbones_dir"], BACKBONE)
print(f"vocab: {voc['path']} ({voc.get('baris')} baris)")
bert_cache_dir = acos_ckpt.backbone_dir(paths["backbones_dir"], BACKBONE)
BIN = os.path.join(bert_cache_dir, "pytorch_model.bin")
if os.path.isfile(BIN):
    BERT_MODEL_SRC = bert_cache_dir
    print(f"checkpoint lokal: {BIN} (Gate-1 numerik penuh di sel training)")
else:
    BERT_MODEL_SRC = "bert-base-uncased"
    print("checkpoint lokal belum ada — bobot diunduh saat from_pretrained (butuh internet).")
    print("Gate-1 struktural (0 encoder-key hilang) tetap dijalankan di sel training.")
print(f"BERT_MODEL_SRC={BERT_MODEL_SRC} (tanpa rekey: prefix bert.* asli)")


In [ ]:
# === SECTION:setup ===
# ============================================================
# 6. EDA torch-free dataset Inggris
# ============================================================
require_vars("step_stage", "acos_root", "DOMAIN", "bert_root")
import acos_en.eda as acos_eda
eda = acos_eda.summarize_domain(acos_root, DOMAIN)
for s in ("train", "dev", "test"):
    d = eda[s]
    print(f"{s}: {d['baris']} kalimat, {d['tuple']} tuple, "
          f"asp_implisit {d['aspek_implisit']}, opi_implisit {d['opini_implisit']}, "
          f"sentimen {d['sentimen']}, kategori teramati {d['n_kategori_teramati']}")
out = os.path.join(bert_root, "build", f"_eda_{DOMAIN}")
print(acos_eda.plot(eda, DOMAIN, os.path.join(out, "plots")))


## Evaluasi final + demo


In [ ]:
# === SECTION:eval ===
# ============================================================
# 10. Evaluasi final + 15 subtask + master_metrics.json
# ============================================================
require_vars("step_stage", "S", "DOMAIN", "tokenizer", "device", "label_list_2",
             "BERT_MODEL_SRC", "N_CATSENTI", "MAX_SEQ_LENGTH", "EVAL_BATCH_SIZE")
import torch
from torch.utils.data import DataLoader, SequentialSampler, TensorDataset
from modeling import CategorySentiClassification
from run_classifier_dataset_utils import processors, convert_examples_to_features2nd
from dataset_utils import read_pair_gold
from eval_metrics import pair_eval
import logging
loggerE = logging.getLogger("acos_bert_eval")

modelE = CategorySentiClassification.from_pretrained(BERT_MODEL_SRC, num_labels=N_CATSENTI[DOMAIN])
best_bin = os.path.join(S, "checkpoints", "step2_best", "pytorch_model.bin")
assert os.path.isfile(best_bin), f"model terbaik tidak ada: {best_bin} — latih Step-2 dulu"
modelE.load_state_dict(torch.load(best_bin, map_location=device))
modelE.to(device); modelE.eval()
processorE = processors["categorysenti"]()
argsE = SimpleNamespace(output_dir=os.path.join(S, "logs"), data_dir=S, domain_type=DOMAIN,
                        max_seq_length=MAX_SEQ_LENGTH, task_name="categorysenti",
                        eval_batch_size=EVAL_BATCH_SIZE)

def _eval_pair_file(pair_path, tag):
    ex = colab_utils.pair_examples_from_file(processorE, pair_path, "test")
    ft = convert_examples_to_features2nd(ex, label_list_2, MAX_SEQ_LENGTH, tokenizer, "categorysenti")
    t = (torch.tensor([f.tokens_len for f in ft], dtype=torch.long),
         torch.tensor([f.aspect_input_ids for f in ft], dtype=torch.long),
         torch.tensor([f.aspect_input_mask for f in ft], dtype=torch.long),
         torch.tensor([f.aspect_segment_ids for f in ft], dtype=torch.long),
         torch.tensor([f.candidate_aspect for f in ft], dtype=torch.long),
         torch.tensor([f.candidate_opinion for f in ft], dtype=torch.long),
         torch.tensor([f.label_id for f in ft], dtype=torch.long))
    dl = DataLoader(TensorDataset(*t), sampler=SequentialSampler(TensorDataset(*t)), batch_size=EVAL_BATCH_SIZE)
    gold = list(read_pair_gold(cs.open(pair_path, encoding="utf-8").readlines(), argsE))
    with colab_utils.SubtaskMetricCapture(loggerE) as cap:
        res = pair_eval("final", argsE, loggerE, tokenizer, modelE, dl, gold, label_list_2, device, "categorysenti", eval_type="test")
    return res, cap.to_frame()

tokS = os.path.join(S, "tokenized_data")
metrics = {"domain": DOMAIN, "num_labels_step2": N_CATSENTI[DOMAIN]}
res_pipe, df_pipe = _eval_pair_file(os.path.join(tokS, f"{DOMAIN}_test_pair_1st.tsv"), "pipeline")
metrics["pipeline_1st"] = {k: (float(v) if isinstance(v, float) else v) for k, v in res_pipe.items()}
res_gold, df_gold = _eval_pair_file(os.path.join(tokS, f"{DOMAIN}_test_pair.tsv"), "gold")
metrics["gold_pair"] = {k: (float(v) if isinstance(v, float) else v) for k, v in res_gold.items()}
print("pipeline (pred Step-1):", metrics["pipeline_1st"])
print("gold pair:", metrics["gold_pair"])

df_pipe.to_csv(os.path.join(S, "csv", "subtask_pipeline.csv"), index=False)
colab_utils.plot_subtask_metrics(df_pipe, os.path.join(S, "plots", "subtask_pipeline.png"),
                                 title=f"{DOMAIN}: 15 subtask (pipeline)")
with open(os.path.join(S, "logs", "master_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print("master_metrics.json + subtask_pipeline.[csv|png] tersimpan")


In [ ]:
# === SECTION:eval ===
# ============================================================
# 11. Demo inferensi 2 kalimat (rest16/laptop)
# ============================================================
require_vars("step_stage", "DOMAIN")
SAMPLES = {"rest16": ["The sushi was fresh but service was slow .",
                      "Great ambience , overpriced food ."],
           "laptop": ["Battery life is great but the keyboard feels cheap .",
                      "Fast shipping , noisy fan ."]}
for s in SAMPLES[DOMAIN]:
    print(">", s)
print("(inferensi penuh memakai model1+modelE terbaik; sel ini contoh pemicu — lihat pred4pipeline.txt sesi untuk hasil batch)")
